In [ ]:
import cv2
import numpy as np
import sys
import matplotlib.pyplot as plt
import os
from omegaconf import OmegaConf

In [ ]:
transformation = "three_c_representation"

config = OmegaConf.load("configs/mobilenet_heatmap.yaml")
width, height, _ = config.training.input_size
base_dir = config.root.data_out + "/" + transformation + "_cropped/train"
files = [f for f in os.listdir(base_dir) if not f.startswith('._') and f.endswith('.png')]

r_values = np.zeros((len(files), height, width), dtype=np.uint8)
g_values = np.zeros((len(files), height, width), dtype=np.uint8)
b_values = np.zeros((len(files), height, width), dtype=np.uint8)

for i, file in enumerate(files):
    img = cv2.imread(os.path.join(base_dir, file))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)  # Convert BGR to RGB
    if img is not None:
        r_values[i] = img[:, :, 0]
        g_values[i] = img[:, :, 1]
        b_values[i] = img[:, :, 2]

# Filter out zero values to avoid skewing the histogram
r_values = r_values[r_values > 0]
g_values = g_values[g_values > 0]
b_values = b_values[b_values > 0]

# Compute histograms manually to get max
bins = 256
r_hist, _ = np.histogram(r_values, bins=bins, range=(0, 256))
g_hist, _ = np.histogram(g_values, bins=bins, range=(0, 256))
b_hist, _ = np.histogram(b_values, bins=bins, range=(0, 256))
max_count = max(r_hist.max(), g_hist.max(), b_hist.max())

# Plot with same scale
fig, axs = plt.subplots(1, 3, figsize=(20,7))
fig.suptitle(transformation, fontsize=16)
axs[0].hist(r_values, bins=bins, color='red', alpha=0.7)
axs[0].set_ylim(0, max_count)
axs[0].set_title('Red Channel Histogram')

axs[1].hist(g_values, bins=bins, color='green', alpha=0.7)
axs[1].set_ylim(0, max_count)
axs[1].set_title('Green Channel Histogram')

axs[2].hist(b_values, bins=bins, color='blue', alpha=0.7)
axs[2].set_ylim(0, max_count)
axs[2].set_title('Blue Channel Histogram')

plt.tight_layout()
plt.show()
